# Ensure there is a symbolic link or the database object in the same directory as this iypnb file

In [ ]:
from pathlib import Path
import duckdb

db_path = Path("../mimic4_note.db").resolve()

con = duckdb.connect(str(db_path))

con.sql("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,mimic4_note,mimiciv_note,discharge,"[note_id, subject_id, hadm_id, note_type, note...","[VARCHAR, INTEGER, INTEGER, VARCHAR, SMALLINT,...",False
1,mimic4_note,mimiciv_note,discharge_detail,"[note_id, subject_id, field_name, field_value,...","[VARCHAR, INTEGER, VARCHAR, VARCHAR, INTEGER]",False
2,mimic4_note,mimiciv_note,radiology,"[note_id, subject_id, hadm_id, note_type, note...","[VARCHAR, INTEGER, INTEGER, VARCHAR, SMALLINT,...",False
3,mimic4_note,mimiciv_note,radiology_detail,"[note_id, subject_id, field_name, field_value,...","[VARCHAR, INTEGER, VARCHAR, VARCHAR, INTEGER]",False


In [33]:
DISCHARGE = "mimic4_note.mimiciv_note.discharge"
DISCHARGE_DETAIL = "mimic4_note.mimiciv_note.discharge_detail"
RADIOLOGY = "mimic4_note.mimiciv_note.radiology"
RADIOLOGY_DETAIL = "mimic4_note.mimiciv_note.radiology_detail"

In [28]:
con.sql(f"""
SELECT *
FROM {DISCHARGE}
LIMIT 5
""").df()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25,2160-11-25 15:09:00,\nName: ___ Unit No: __...


In [34]:
con.sql(f"""
SELECT *
FROM {DISCHARGE_DETAIL}
LIMIT 5
""").df()

,note_id,subject_id,field_name,field_value,field_ordinal
0,10000032-DS-21,10000032,author,___,1
1,10000032-DS-22,10000032,author,___,1
2,10000032-DS-23,10000032,author,___,1
3,10000032-DS-24,10000032,author,___,1
4,10000084-DS-17,10000084,author,___,1


Confirms that dischrage detail contains mainly metadata about the note, and not the note text itself which may be less useful for NLP tasks. The note text is in the discharge table.

In [29]:
for tbl in [
    "discharge",
    "discharge_detail",
    "radiology",
    "radiology_detail"
]:
    print(tbl)
    print(
        con.sql(f"""
        SELECT COUNT(*)
        FROM mimic4_note.mimiciv_note.{tbl}
        """).fetchone()[0]
    )

discharge
331793
discharge_detail
186138
radiology
2321355
radiology_detail
6046121


In [37]:
# Check note lengths:

con.sql(f"""
SELECT
    AVG(length(text)) AS avg_chars,
    MIN(length(text)) AS min_chars,
    MAX(length(text)) AS max_chars
FROM {DISCHARGE}
""").df()

,avg_chars,min_chars,max_chars
0,10550.96027,353,60381


In [38]:
note = con.sql(f"""
SELECT text
FROM {DISCHARGE}
LIMIT 1
""").fetchone()[0]

print(note)

 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
distension over past week.  
Pt reports self-discontinuing lasix and spirnolactone ___ weeks 
ago, because she feels like "they don't do anything" and that 
she "doesn't want to put more chemicals in her." She does not 
follow Na-restricted diets. In the past week, she notes that she 
has been having worsening abd distension and discomfort. She 
denies ___ edema, or SOB, or orthopnea. She denies f/c/n/v, d/c, 
dysuria. She had food poisoning a week ago from eating stale 
cake (n/v 20 min after fo

# Identifying Note Structural Stratification

In [ ]:
# Check note types:

con.sql(f"""
SELECT
    note_type,
    COUNT(*) AS n
FROM {DISCHARGE}
GROUP BY note_type
ORDER BY n DESC
""").df()

,note_type,n
0,DS,331793


In [ ]:
# comparing number of rows to number of unique admissions to check if there are multiple notes per admission:
con.sql(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT hadm_id) AS n_admissions
FROM {DISCHARGE}
""").df()

,n_rows,n_admissions
0,331793,331793


In [48]:
# confirmation
con.sql(f"""
SELECT
    MAX(cnt) AS max_notes_per_admission
FROM (
    SELECT hadm_id, COUNT(*) AS cnt
    FROM {DISCHARGE}
    GROUP BY hadm_id
)
""").df()

,max_notes_per_admission
0,1


Examination of the discharge notes table showed that all records were labeled as discharge summaries (DS), with no evidence of discharge summary addenda (AD), operating room notes (OP), or operating room note addenda (AO) in the imported dataset. Furthermore, each hospitalization (hadm_id) was associated with exactly one discharge summary, indicating a one-to-one relationship between admissions and discharge notes. However, individual patients (subject_id) could have multiple discharge summaries across different hospitalizations. This structure simplifies downstream NLP analyses because each admission can be represented by a single discharge note without the need to aggregate multiple notes within the same encounter. Depending on the study objective, analyses may still consider the number of discharge summaries per patient as a proxy for healthcare utilization or longitudinal clinical history.

# NLP Plan

the modeling/embedding process can be expensive, so I decided to start with a smaller subset of the data (e.g. 10k notes) to iterate quickly on the modeling process and then scale up to the full dataset once I have a working pipeline. 

* create  an admission-level dataset where each row corresponds to a unique hadm_id and contains the discharge summary text for that admission

In [50]:
con.sql(f"""
SELECT
    COUNT(*) AS n_notes,
    COUNT(DISTINCT subject_id) AS n_subjects
FROM {DISCHARGE}
""").df()

,n_notes,n_subjects
0,331793,145914


In [ ]:
# checking upper bound of number of notes per patient:
con.sql(f"""
SELECT
    subject_id,
    COUNT(*) AS n_notes
FROM {DISCHARGE}
GROUP BY subject_id
ORDER BY n_notes DESC
LIMIT 20
""").df()

,subject_id,n_notes
0,17517983,89
1,13297743,89
2,12468016,85
3,11965254,83
4,13475033,81
5,19133405,76
6,11413236,76
7,10577647,75
8,11296936,72
9,18284271,71


In [ ]:
# proof of concept for subsetting

nlp_subset = con.sql(f"""
SELECT *
FROM {DISCHARGE}
USING SAMPLE reservoir(5000 ROWS) REPEATABLE (321)
""").df()

nlp_subset["subject_id"].value_counts().describe()

count    4833.000000
mean        1.034554
std         0.199974
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: count, dtype: float64

In [64]:
counts = nlp_subset["subject_id"].value_counts()

print("Unique patients:", counts.shape[0])
print("Max notes per patient:", counts.max())
print("Patients with >1 note:", (counts > 1).sum())

Unique patients: 4833
Max notes per patient: 5
Patients with >1 note: 154


# Create 30 day readmission label

In [65]:
ADMISSIONS = "/home/Max_Vo/mimic/hosp/admissions.csv.gz"

con.sql(f"""
CREATE OR REPLACE VIEW admissions AS
SELECT *
FROM read_csv_auto('{ADMISSIONS}')
""")

In [66]:
con.sql("""
DESCRIBE admissions
""").df()

,column_name,column_type,null,key,default,extra
0,subject_id,BIGINT,YES,None,None,None
1,hadm_id,BIGINT,YES,None,None,None
2,admittime,TIMESTAMP,YES,None,None,None
3,dischtime,TIMESTAMP,YES,None,None,None
4,deathtime,TIMESTAMP,YES,None,None,None
5,admission_type,VARCHAR,YES,None,None,None
6,admit_provider_id,VARCHAR,YES,None,None,None
7,admission_location,VARCHAR,YES,None,None,None
8,discharge_location,VARCHAR,YES,None,None,None
9,insurance,VARCHAR,YES,None,None,None


In [70]:
readmit_labels = con.sql("""
WITH admissions_ordered AS (
    SELECT
        subject_id,
        hadm_id,
        admittime,
        dischtime,
        hospital_expire_flag,

        LEAD(admittime) OVER (
            PARTITION BY subject_id
            ORDER BY admittime
        ) AS next_admittime

    FROM admissions
)

SELECT
    subject_id,
    hadm_id,

    DATE_DIFF(
        'day',
        dischtime,
        next_admittime
    ) AS days_to_next_admission,

    CASE
        WHEN next_admittime IS NOT NULL
         AND DATE_DIFF(
                'day',
                dischtime,
                next_admittime
             ) <= 30
        THEN 1
        ELSE 0
    END AS readmit_30d

FROM admissions_ordered
WHERE hospital_expire_flag = 0
""")

the 30 day readmission label:

* Excludes patients who died in hospital.
* Uses the next admission chronologically.
* Creates a binary label.

In [ ]:
# "What proportion of admissions were followed by a readmission within 30 days?"

readmit_labels.df()["readmit_30d"].value_counts(normalize=True)

readmit_30d
0    0.793666
1    0.206334
Name: proportion, dtype: float64

Approximately 20% of admissions were followed by a readmission within 30 days, which is consistent with prior literature on hospital readmissions. This suggests that the dataset is representative of typical readmission rates and should provide sufficient signal for modeling efforts.

## Combine Sampling Strategy with Readmission Labeling

In [98]:
con.sql(f"""
CREATE OR REPLACE TABLE nlp_subset_readmit AS

WITH admissions_ordered AS (
    SELECT
        subject_id,
        hadm_id,
        dischtime,
        hospital_expire_flag,
        LEAD(admittime) OVER (
            PARTITION BY subject_id
            ORDER BY admittime
        ) AS next_admittime
    FROM admissions
),

readmit_labels AS (
    SELECT
        subject_id,
        hadm_id,
        DATE_DIFF('day', dischtime, next_admittime) AS days_to_next_admission,
        CASE
            WHEN next_admittime IS NOT NULL
             AND DATE_DIFF('day', dischtime, next_admittime) <= 30
            THEN 1
            ELSE 0
        END AS readmit_30d
    FROM admissions_ordered
    WHERE hospital_expire_flag = 0
),

labeled_discharge AS (
    SELECT
        CAST(d.note_id AS VARCHAR) AS note_id,
        CAST(d.subject_id AS INTEGER) AS subject_id,
        CAST(d.hadm_id AS INTEGER) AS hadm_id,
        CAST(d.charttime AS TIMESTAMP) AS charttime,
        CAST(d.storetime AS TIMESTAMP) AS storetime,
        CAST(d.text AS VARCHAR) AS text,
        CAST(r.days_to_next_admission AS INTEGER) AS days_to_next_admission,
        CAST(r.readmit_30d AS SMALLINT) AS readmit_30d
    FROM {DISCHARGE} d
    INNER JOIN readmit_labels r
        ON d.subject_id = r.subject_id
       AND d.hadm_id = r.hadm_id
)

SELECT *
FROM labeled_discharge
USING SAMPLE reservoir(5000 ROWS) REPEATABLE (321)
""")

In [99]:
con.sql("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,mimic4_note,main,admissions,"[subject_id, hadm_id, admittime, dischtime, de...","[BIGINT, BIGINT, TIMESTAMP, TIMESTAMP, TIMESTA...",False
1,mimic4_note,main,nlp_subset_readmit,"[note_id, subject_id, hadm_id, charttime, stor...","[VARCHAR, INTEGER, INTEGER, TIMESTAMP, TIMESTA...",False
2,mimic4_note,mimiciv_note,discharge,"[note_id, subject_id, hadm_id, note_type, note...","[VARCHAR, INTEGER, INTEGER, VARCHAR, SMALLINT,...",False
3,mimic4_note,mimiciv_note,discharge_detail,"[note_id, subject_id, field_name, field_value,...","[VARCHAR, INTEGER, VARCHAR, VARCHAR, INTEGER]",False
4,mimic4_note,mimiciv_note,radiology,"[note_id, subject_id, hadm_id, note_type, note...","[VARCHAR, INTEGER, INTEGER, VARCHAR, SMALLINT,...",False
5,mimic4_note,mimiciv_note,radiology_detail,"[note_id, subject_id, field_name, field_value,...","[VARCHAR, INTEGER, VARCHAR, VARCHAR, INTEGER]",False


In [101]:
nlp_subset_readmit = con.sql("""
SELECT *
FROM nlp_subset_readmit
""").df()

In [102]:
nlp_subset_readmit["readmit_30d"].value_counts(normalize=True)

readmit_30d
0    0.779
1    0.221
Name: proportion, dtype: float64

In [103]:
nlp_subset_readmit["subject_id"].value_counts().describe()

count    4812.000000
mean        1.039069
std         0.222724
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: count, dtype: float64

A reproducible reservoir sample was used to obtain a computationally manageable subset of discharge summaries. While the target sample size was 5,000 notes, the final sample contained 4,812 observations. This slight deviation reflects the behavior of the sampling operation after cohort construction and does not materially affect the downstream analyses.

In [106]:
con.sql("DESCRIBE nlp_subset_readmit").df()

,column_name,column_type,null,key,default,extra
0,note_id,VARCHAR,YES,None,None,None
1,subject_id,INTEGER,YES,None,None,None
2,hadm_id,INTEGER,YES,None,None,None
3,charttime,TIMESTAMP,YES,None,None,None
4,storetime,TIMESTAMP,YES,None,None,None
5,text,VARCHAR,YES,None,None,None
6,days_to_next_admission,INTEGER,YES,None,None,None
7,readmit_30d,SMALLINT,YES,None,None,None


In [107]:
con.sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_subjects,
    AVG(readmit_30d) AS readmission_prevalence
FROM nlp_subset_readmit
""").df()

,n_rows,n_subjects,readmission_prevalence
0,5000,4812,0.221


In [108]:
con.close()

# Cohort Summary

To create a computationally tractable dataset for model development, discharge summaries from MIMIC-IV Note were linked to hospitalization records in MIMIC-IV Hosp through subject_id and hadm_id. A binary 30-day readmission outcome was generated by identifying the next admission for each patient and calculating the interval between discharge and subsequent admission. Hospitalizations resulting in in-hospital death were excluded. The resulting admission-level dataset was then randomly subsampled using reproducible reservoir sampling (REPEATABLE (321)), yielding 5000 (4812 unique subjects) discharge summaries with associated readmission labels. Because each hospitalization was associated with a single discharge summary, each row in the final cohort represented one admission-note pair, suitable for downstream transformer-based text embedding and predictive modeling workflows.